# Classification Reliability Heat Maps

Two complementary reliability checks for the primary pathway
classification, both expressed as a **label-retention fraction**: the
fraction of pathways in a given full-dataset primary class that keep the
same primary label when the classification is repeated under some form
of perturbation.

- **Part A -- Bootstrap convergence.** Classification thresholds are
  learned from a random subset of `n` pathways (`n` swept from 10% to
  100% of the dataset) and applied to the full dataset. This tests
  sensitivity to *which* pathways were used to define the thresholds.
- **Part B -- Combined bootstrap + threshold perturbation.** As above,
  but the quantile levels used to define the thresholds (e.g. the 75th
  percentile) are additionally perturbed by a few percentile points, and
  the recovery cutoffs (`RE` thresholds) are jittered too. This is a
  stronger test that also probes sensitivity to the *exact* threshold
  definitions, not just the data used to set them.

For a class `c`, the retention score at sample size `n` is

$$
S_{n,c} = \frac{1}{BP N_c} \sum_{b=1}^{B} \sum_{p=1}^{P} \sum_{k \in c}
\mathbf{1}\!\left[ L_k^{(b,p,n)} = L_k^{\mathrm{ref}} \right]
$$

where $L_k^{\mathrm{ref}}$ is pathway $k$'s primary label under the
full-dataset reference classification, $L_k^{(b,p,n)}$ its label under
bootstrap repeat $b$ and (for Part B) threshold-perturbation repeat $p$,
$B$ is the number of bootstrap repeats, $P$ the number of threshold
perturbations, and $N_c$ the number of pathways in reference class $c$.
Values close to 1 indicate a class whose membership is essentially
unaffected by finite sampling and threshold choice.


## Part A -- Bootstrap convergence heat map

Learn thresholds from a random subset of pathways, classify the full dataset, and compare to the full-dataset reference labels.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# USER SETTINGS
# ============================================================

ROOT_DIR = Path("./example_data")  # TODO: point this at a directory containing path_couplings_timeseries.csv files for one GB structure

ORDER_DESCRIPTOR_FILE = ROOT_DIR / "paths_order_descriptors_with_archetype_labels.csv"

OUT_DIR = Path("./results")  # TODO: point this at the folder where figures/CSVs from this notebook should be written
OUT_DIR.mkdir(exist_ok=True, parents=True)

EPS = 1e-12

# Same options as in your main classification code
INCLUDE_DIFFUSE_AS_5TH_LABEL = False
FLAT_COUNTS_AS_NON_RECOVERING = True

# Bootstrap settings
N_BOOTSTRAP = 300

# Use either fractions or fixed numbers.
# The code will convert these fractions to number of sampled pathways.
SAMPLE_FRACTIONS = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 1.00]

RANDOM_SEED = 42

PRIMARY_ORDER = [
    "Diffuse/weakly structured",
    "Source-limited",
    "GB-crossing (recovering)",
    "GB-blocked (non-recovering)",
    "Target-limited",
]

if not INCLUDE_DIFFUSE_AS_5TH_LABEL:
    PRIMARY_ORDER = [
        "Source-limited",
        "GB-crossing (recovering)",
        "GB-blocked (non-recovering)",
        "Target-limited",
    ]


# ============================================================
# CLASSIFICATION FUNCTIONS
# Same logic as your original code, but written so thresholds
# can be learned from a bootstrap subset and applied to all paths.
# ============================================================

def get_thresholds_from_reference(df_ref):
    """
    Quantile thresholds computed from a reference dataset.
    This can be the full dataset or a bootstrap subset.
    """
    thresholds = {
        "pE_lo": float(np.percentile(df_ref["pE"], 20)),
        "pE_hi": float(np.percentile(df_ref["pE"], 80)),

        "E_dip_q75": float(np.percentile(df_ref["E_dip"], 75)),
        "C_dip_q75": float(np.percentile(df_ref["C_dip"], 75)),

        "rJ_q25": float(np.percentile(df_ref["rJ"], 25)),
        "rC_q25": float(np.percentile(df_ref["rC"], 25)),

        "Lweak_E_q75": float(np.percentile(df_ref["Lweak_E"], 75)),
        "Lweak_C_q75": float(np.percentile(df_ref["Lweak_C"], 75)),

        "drop_E_q75": float(np.percentile(df_ref["drop_E"], 75)),
        "rise_E_q75": float(np.percentile(df_ref["rise_E"], 75)),

        "drop_C_q75": float(np.percentile(df_ref["drop_C"], 75)),
        "rise_C_q75": float(np.percentile(df_ref["rise_C"], 75)),

        "rec_hi": 1.05,
        "rec_lo": 0.95,
    }

    return thresholds


def apply_archetype_labels(df, thresholds):
    """
    Apply order-aware labels using externally provided thresholds.
    """
    out = df.copy()

    loc_labels = []
    cause_labels = []
    shape_labels = []
    rec_labels = []
    combined_labels = []

    for _, r in out.iterrows():

        # -----------------------------
        # Location label
        # -----------------------------
        if r["pE"] <= thresholds["pE_lo"]:
            label_location = "source-limited"
        elif r["pE"] >= thresholds["pE_hi"]:
            label_location = "target-limited"
        else:
            label_location = "GB-limited"

        # -----------------------------
        # Cause label
        # -----------------------------
        coupling_signal = (
            (r["E_dip"] >= thresholds["E_dip_q75"])
            or (r["drop_E"] >= thresholds["drop_E_q75"])
            or (r["Lweak_E"] >= thresholds["Lweak_E_q75"])
            or (r["rJ"] <= thresholds["rJ_q25"])
        )

        coherence_signal = (
            (r["C_dip"] >= thresholds["C_dip_q75"])
            or (r["drop_C"] >= thresholds["drop_C_q75"])
            or (r["Lweak_C"] >= thresholds["Lweak_C_q75"])
            or (r["rC"] <= thresholds["rC_q25"])
        )

        if coupling_signal and not coherence_signal:
            label_cause = "coupling-limited"
        elif coherence_signal and not coupling_signal:
            label_cause = "coherence-limited"
        else:
            label_cause = "mixed-limited"

        # -----------------------------
        # Shape label
        # -----------------------------
        sharp = (
            ((r["drop_E"] >= thresholds["drop_E_q75"]) and (r["rise_E"] >= thresholds["rise_E_q75"]))
            or ((r["drop_C"] >= thresholds["drop_C_q75"]) and (r["rise_C"] >= thresholds["rise_C_q75"]))
        )

        extended = (
            (r["Lweak_E"] >= thresholds["Lweak_E_q75"])
            or (r["Lweak_C"] >= thresholds["Lweak_C_q75"])
        )

        if sharp and extended:
            label_shape = "sharp+extended"
        elif sharp:
            label_shape = "sharp"
        elif extended:
            label_shape = "extended"
        else:
            label_shape = "diffuse"

        # -----------------------------
        # Recovery label
        # -----------------------------
        if r["RE"] > thresholds["rec_hi"]:
            label_recovery = "recovering"
        elif r["RE"] < thresholds["rec_lo"]:
            label_recovery = "degrading"
        else:
            label_recovery = "flat"

        label_combined = f"{label_location}, {label_cause}, {label_shape}, {label_recovery}"

        loc_labels.append(label_location)
        cause_labels.append(label_cause)
        shape_labels.append(label_shape)
        rec_labels.append(label_recovery)
        combined_labels.append(label_combined)

    out["label_location"] = loc_labels
    out["label_cause"] = cause_labels
    out["label_shape"] = shape_labels
    out["label_recovery"] = rec_labels
    out["label_combined"] = combined_labels

    return out


def add_primary_label(
    df,
    include_diffuse_as_5th=True,
    flat_counts_as_nonrecovering=True,
):
    """
    Same primary-label merging logic as your original notebook.
    """
    out = df.copy()

    if flat_counts_as_nonrecovering:
        nonrec = out["label_recovery"].isin(["degrading", "flat"])
    else:
        nonrec = out["label_recovery"].eq("degrading")

    out["label_primary"] = "Unassigned"

    if include_diffuse_as_5th:
        out.loc[out["label_shape"].eq("diffuse"), "label_primary"] = "Diffuse/weakly structured"

    out.loc[out["label_location"].eq("source-limited"), "label_primary"] = "Source-limited"
    out.loc[out["label_location"].eq("target-limited"), "label_primary"] = "Target-limited"

    gb = out["label_location"].eq("GB-limited")

    out.loc[
        gb & out["label_recovery"].eq("recovering"),
        "label_primary"
    ] = "GB-crossing (recovering)"

    out.loc[
        gb & nonrec,
        "label_primary"
    ] = "GB-blocked (non-recovering)"

    return out


def classify_with_reference_thresholds(df_all, df_threshold_reference):
    """
    Learn thresholds from df_threshold_reference and apply them to df_all.
    """
    thresholds = get_thresholds_from_reference(df_threshold_reference)
    labeled = apply_archetype_labels(df_all, thresholds)
    labeled = add_primary_label(
        labeled,
        include_diffuse_as_5th=INCLUDE_DIFFUSE_AS_5TH_LABEL,
        flat_counts_as_nonrecovering=FLAT_COUNTS_AS_NON_RECOVERING,
    )
    return labeled


# ============================================================
# RELIABILITY / CONVERGENCE ANALYSIS
# ============================================================

def bootstrap_classification_convergence(df):
    """
    For increasing sample size:
    1. sample pathways,
    2. learn thresholds from the sampled subset,
    3. classify the full dataset using those thresholds,
    4. compare labels to the full-dataset reference labels.

    Returns
    -------
    stability_matrix : DataFrame
        Rows = number of sampled pathways
        Columns = reference primary labels
        Values = mean label retention fraction
    """

    rng = np.random.default_rng(RANDOM_SEED)

    n_total = len(df)

    # Full-dataset reference classification
    reference_labeled = classify_with_reference_thresholds(df, df)
    reference_labels = reference_labeled["label_primary"].to_numpy()

    # Save reference labels
    reference_labeled.to_csv(
        OUT_DIR / "reference_full_dataset_labels.csv",
        index=False,
    )

    sample_sizes = []
    for frac in SAMPLE_FRACTIONS:
        n_sample = int(round(frac * n_total))
        n_sample = max(5, min(n_sample, n_total))
        sample_sizes.append(n_sample)

    sample_sizes = sorted(set(sample_sizes))

    rows = []

    for n_sample in sample_sizes:
        print(f"Running bootstrap convergence for n_sample = {n_sample}/{n_total}")

        stability_by_label = {label: [] for label in PRIMARY_ORDER}

        for _ in range(N_BOOTSTRAP):

            sampled_idx = rng.choice(
                np.arange(n_total),
                size=n_sample,
                replace=False,
            )

            df_sample = df.iloc[sampled_idx].copy()

            bootstrap_labeled = classify_with_reference_thresholds(df, df_sample)
            bootstrap_labels = bootstrap_labeled["label_primary"].to_numpy()

            for label in PRIMARY_ORDER:
                mask = reference_labels == label

                if np.sum(mask) == 0:
                    stability = np.nan
                else:
                    stability = np.mean(bootstrap_labels[mask] == reference_labels[mask])

                stability_by_label[label].append(stability)

        row = {
            "n_sampled_paths": n_sample,
            "sample_fraction": n_sample / n_total,
        }

        for label in PRIMARY_ORDER:
            row[label] = np.nanmean(stability_by_label[label])

        rows.append(row)

    stability_df = pd.DataFrame(rows)
    return stability_df


# ============================================================
# PLOTTING FUNCTION
# ============================================================

def plot_convergence_heatmap(stability_df):
    """
    One heat map:
    rows = number of sampled pathways used for threshold estimation
    columns = primary labels
    color = label retention relative to full-dataset classification
    """

    plot_df = stability_df.set_index("n_sampled_paths")[PRIMARY_ORDER]

    fig_width = max(8, 1.8 * len(PRIMARY_ORDER))
    fig_height = max(5, 0.45 * len(plot_df))

    fig, ax = plt.subplots(figsize=(fig_width, fig_height))

    im = ax.imshow(
        plot_df.values,
        aspect="auto",
        vmin=0.0,
        vmax=1.0,
    )

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Primary-label retention fraction")

    ax.set_title("Bootstrap convergence of pathway classification", pad=14)
    ax.set_xlabel("Reference primary label")
    ax.set_ylabel("Number of pathways used to define thresholds")

    ax.set_xticks(np.arange(len(plot_df.columns)))
    ax.set_xticklabels(plot_df.columns, rotation=35, ha="right")

    ax.set_yticks(np.arange(len(plot_df.index)))
    ax.set_yticklabels(plot_df.index)

    # Add values inside cells
    for i in range(plot_df.shape[0]):
        for j in range(plot_df.shape[1]):
            value = plot_df.iloc[i, j]
            if not np.isnan(value):
                ax.text(
                    j,
                    i,
                    f"{value:.2f}",
                    ha="center",
                    va="center",
                    fontsize=8,
                )

    fig.tight_layout()

    out_png = OUT_DIR / "classification_convergence_heatmap.png"
    out_pdf = OUT_DIR / "classification_convergence_heatmap.pdf"

    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    fig.savefig(out_pdf, bbox_inches="tight")

    plt.show()

    print(f"\nSaved heat map:")
    print(out_png)
    print(out_pdf)


# ============================================================
# RUN
# ============================================================

df = pd.read_csv(ORDER_DESCRIPTOR_FILE)

required_cols = [
    "pE", "E_dip", "C_dip",
    "rJ", "rC",
    "Lweak_E", "Lweak_C",
    "drop_E", "rise_E",
    "drop_C", "rise_C",
    "RE",
]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required descriptor columns: {missing}")

stability_df = bootstrap_classification_convergence(df)

stability_df.to_csv(
    OUT_DIR / "classification_convergence_values.csv",
    index=False,
)

plot_convergence_heatmap(stability_df)

print("\nConvergence values:")
print(stability_df.round(3).to_string(index=False))

## Part B -- Combined bootstrap + threshold-perturbation heat map

As Part A, but also randomly perturbs the quantile levels and recovery cutoffs used to define the thresholds on each trial.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# USER SETTINGS
# ============================================================

ROOT_DIR = Path("./example_data")  # TODO: point this at a directory containing path_couplings_timeseries.csv files for one GB structure

ORDER_DESCRIPTOR_FILE = ROOT_DIR / "paths_order_descriptors_with_archetype_labels.csv"

OUT_DIR = Path("./results")  # TODO: point this at the folder where figures/CSVs from this notebook should be written
OUT_DIR.mkdir(exist_ok=True)

# Same settings as your original classification
INCLUDE_DIFFUSE_AS_5TH_LABEL = False
FLAT_COUNTS_AS_NON_RECOVERING = True

# Bootstrap settings
N_BOOTSTRAP = 300

# Threshold perturbations per bootstrap sample
N_PERTURB = 30

# Fraction of total pathways used to define thresholds
SAMPLE_FRACTIONS = [
    0.10, 0.20, 0.30, 0.40, 0.50,
    0.60, 0.70, 0.80, 0.90, 1.00
]

RANDOM_SEED = 42

# Quantile perturbation widths
# Example:
# q20 becomes randomly sampled from q15 to q25
# q75 becomes randomly sampled from q70 to q80
# q80 becomes randomly sampled from q75 to q85
Q_PERTURB_WIDTH = 5.0

# Recovery threshold perturbation
# rec_hi = 1.05 +/- REC_PERTURB
# rec_lo = 0.95 +/- REC_PERTURB
REC_PERTURB = 0.02

PRIMARY_ORDER = [
    "Source-limited",
    "GB-crossing (recovering)",
    "GB-blocked (non-recovering)",
    "Target-limited",
]

if INCLUDE_DIFFUSE_AS_5TH_LABEL:
    PRIMARY_ORDER = [
        "Diffuse/weakly structured",
        "Source-limited",
        "GB-crossing (recovering)",
        "GB-blocked (non-recovering)",
        "Target-limited",
    ]


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def safe_percentile(x, q):
    """
    Percentile with q clipped between 0 and 100.
    """
    q = np.clip(q, 0.0, 100.0)
    return float(np.percentile(x, q))


def draw_perturbed_quantile(base_q, rng, width=5.0):
    """
    Draw a perturbed quantile level around base_q.
    Example: base_q = 75 and width = 5 gives random q in [70, 80].
    """
    q = rng.uniform(base_q - width, base_q + width)
    return float(np.clip(q, 0.0, 100.0))


def get_thresholds_from_reference(
    df_ref,
    rng=None,
    perturb_quantiles=False,
    q_width=5.0,
    perturb_recovery=False,
    rec_width=0.02,
):
    """
    Computes classification thresholds from a reference subset.

    If perturb_quantiles=True, the quantile levels themselves are perturbed.
    This directly tests robustness of your quantile-based classification.
    """

    if rng is None:
        rng = np.random.default_rng(0)

    # -----------------------------
    # Quantile levels
    # -----------------------------
    if perturb_quantiles:
        q20 = draw_perturbed_quantile(20, rng, q_width)
        q25 = draw_perturbed_quantile(25, rng, q_width)
        q75 = draw_perturbed_quantile(75, rng, q_width)
        q80 = draw_perturbed_quantile(80, rng, q_width)
    else:
        q20 = 20.0
        q25 = 25.0
        q75 = 75.0
        q80 = 80.0

    # -----------------------------
    # Recovery thresholds
    # -----------------------------
    if perturb_recovery:
        rec_hi = 1.05 + rng.uniform(-rec_width, rec_width)
        rec_lo = 0.95 + rng.uniform(-rec_width, rec_width)

        # Safety: keep rec_lo below rec_hi
        if rec_lo >= rec_hi:
            rec_lo = 0.95
            rec_hi = 1.05
    else:
        rec_hi = 1.05
        rec_lo = 0.95

    thresholds = {
        # Location thresholds
        "pE_lo": safe_percentile(df_ref["pE"], q20),
        "pE_hi": safe_percentile(df_ref["pE"], q80),

        # Dip thresholds
        "E_dip_q75": safe_percentile(df_ref["E_dip"], q75),
        "C_dip_q75": safe_percentile(df_ref["C_dip"], q75),

        # Bottleneck depth thresholds
        "rJ_q25": safe_percentile(df_ref["rJ"], q25),
        "rC_q25": safe_percentile(df_ref["rC"], q25),

        # Weak-region thresholds
        "Lweak_E_q75": safe_percentile(df_ref["Lweak_E"], q75),
        "Lweak_C_q75": safe_percentile(df_ref["Lweak_C"], q75),

        # Drop/rise thresholds
        "drop_E_q75": safe_percentile(df_ref["drop_E"], q75),
        "rise_E_q75": safe_percentile(df_ref["rise_E"], q75),
        "drop_C_q75": safe_percentile(df_ref["drop_C"], q75),
        "rise_C_q75": safe_percentile(df_ref["rise_C"], q75),

        # Recovery thresholds
        "rec_hi": rec_hi,
        "rec_lo": rec_lo,

        # Store quantiles for later checking
        "q20_used": q20,
        "q25_used": q25,
        "q75_used": q75,
        "q80_used": q80,
    }

    return thresholds


def apply_archetype_labels(df, thresholds):
    """
    Applies order-aware labels using externally supplied thresholds.
    This follows your original classification logic.
    """

    out = df.copy()

    loc_labels = []
    cause_labels = []
    shape_labels = []
    rec_labels = []
    combined_labels = []

    for _, r in out.iterrows():

        # ========================================================
        # Location
        # ========================================================
        if r["pE"] <= thresholds["pE_lo"]:
            label_location = "source-limited"
        elif r["pE"] >= thresholds["pE_hi"]:
            label_location = "target-limited"
        else:
            label_location = "GB-limited"

        # ========================================================
        # Cause
        # ========================================================
        coupling_signal = (
            (r["E_dip"] >= thresholds["E_dip_q75"])
            or (r["drop_E"] >= thresholds["drop_E_q75"])
            or (r["Lweak_E"] >= thresholds["Lweak_E_q75"])
            or (r["rJ"] <= thresholds["rJ_q25"])
        )

        coherence_signal = (
            (r["C_dip"] >= thresholds["C_dip_q75"])
            or (r["drop_C"] >= thresholds["drop_C_q75"])
            or (r["Lweak_C"] >= thresholds["Lweak_C_q75"])
            or (r["rC"] <= thresholds["rC_q25"])
        )

        if coupling_signal and not coherence_signal:
            label_cause = "coupling-limited"
        elif coherence_signal and not coupling_signal:
            label_cause = "coherence-limited"
        else:
            label_cause = "mixed-limited"

        # ========================================================
        # Shape
        # ========================================================
        sharp = (
            (
                (r["drop_E"] >= thresholds["drop_E_q75"])
                and (r["rise_E"] >= thresholds["rise_E_q75"])
            )
            or
            (
                (r["drop_C"] >= thresholds["drop_C_q75"])
                and (r["rise_C"] >= thresholds["rise_C_q75"])
            )
        )

        extended = (
            (r["Lweak_E"] >= thresholds["Lweak_E_q75"])
            or (r["Lweak_C"] >= thresholds["Lweak_C_q75"])
        )

        if sharp and extended:
            label_shape = "sharp+extended"
        elif sharp:
            label_shape = "sharp"
        elif extended:
            label_shape = "extended"
        else:
            label_shape = "diffuse"

        # ========================================================
        # Recovery
        # ========================================================
        if r["RE"] > thresholds["rec_hi"]:
            label_recovery = "recovering"
        elif r["RE"] < thresholds["rec_lo"]:
            label_recovery = "degrading"
        else:
            label_recovery = "flat"

        label_combined = (
            f"{label_location}, "
            f"{label_cause}, "
            f"{label_shape}, "
            f"{label_recovery}"
        )

        loc_labels.append(label_location)
        cause_labels.append(label_cause)
        shape_labels.append(label_shape)
        rec_labels.append(label_recovery)
        combined_labels.append(label_combined)

    out["label_location"] = loc_labels
    out["label_cause"] = cause_labels
    out["label_shape"] = shape_labels
    out["label_recovery"] = rec_labels
    out["label_combined"] = combined_labels

    return out


def add_primary_label(
    df,
    include_diffuse_as_5th=True,
    flat_counts_as_nonrecovering=True,
):
    """
    Merges location/recovery/shape labels into primary labels.
    This follows your original primary-label logic.
    """

    out = df.copy()

    if flat_counts_as_nonrecovering:
        nonrec = out["label_recovery"].isin(["degrading", "flat"])
    else:
        nonrec = out["label_recovery"].eq("degrading")

    out["label_primary"] = "Unassigned"

    # Optional diffuse class
    if include_diffuse_as_5th:
        out.loc[
            out["label_shape"].eq("diffuse"),
            "label_primary"
        ] = "Diffuse/weakly structured"

    # Source/target classes overwrite diffuse if location is source/target
    out.loc[
        out["label_location"].eq("source-limited"),
        "label_primary"
    ] = "Source-limited"

    out.loc[
        out["label_location"].eq("target-limited"),
        "label_primary"
    ] = "Target-limited"

    # GB classes
    gb = out["label_location"].eq("GB-limited")

    out.loc[
        gb & out["label_recovery"].eq("recovering"),
        "label_primary"
    ] = "GB-crossing (recovering)"

    out.loc[
        gb & nonrec,
        "label_primary"
    ] = "GB-blocked (non-recovering)"

    return out


def classify_with_threshold_reference(
    df_all,
    df_threshold_reference,
    rng=None,
    perturb_quantiles=False,
    perturb_recovery=False,
):
    """
    Learns thresholds from df_threshold_reference and applies classification
    to df_all.
    """

    thresholds = get_thresholds_from_reference(
        df_threshold_reference,
        rng=rng,
        perturb_quantiles=perturb_quantiles,
        q_width=Q_PERTURB_WIDTH,
        perturb_recovery=perturb_recovery,
        rec_width=REC_PERTURB,
    )

    labeled = apply_archetype_labels(df_all, thresholds)

    labeled = add_primary_label(
        labeled,
        include_diffuse_as_5th=INCLUDE_DIFFUSE_AS_5TH_LABEL,
        flat_counts_as_nonrecovering=FLAT_COUNTS_AS_NON_RECOVERING,
    )

    return labeled, thresholds


# ============================================================
# MAIN RELIABILITY ANALYSIS
# ============================================================

def run_combined_bootstrap_threshold_analysis(df):
    """
    Combined convergence and reliability analysis.

    For each sample size:
        - randomly sample pathways to define thresholds
        - perturb quantile levels
        - classify full dataset
        - compare to full-dataset reference labels

    Returns
    -------
    stability_mean_df : DataFrame
        Mean retention fraction per primary label.

    stability_std_df : DataFrame
        Standard deviation of retention fraction per primary label.

    all_results_df : DataFrame
        Long-format results for every bootstrap and perturbation run.
    """

    rng = np.random.default_rng(RANDOM_SEED)

    n_total = len(df)

    # ========================================================
    # Full-dataset reference classification
    # No perturbation, full dataset thresholds.
    # ========================================================
    reference_labeled, reference_thresholds = classify_with_threshold_reference(
        df_all=df,
        df_threshold_reference=df,
        rng=rng,
        perturb_quantiles=False,
        perturb_recovery=False,
    )

    reference_labels = reference_labeled["label_primary"].to_numpy()

    reference_labeled.to_csv(
        OUT_DIR / "reference_full_dataset_labels.csv",
        index=False,
    )

    pd.DataFrame([reference_thresholds]).to_csv(
        OUT_DIR / "reference_full_dataset_thresholds.csv",
        index=False,
    )

    # Check class counts
    reference_counts = reference_labeled["label_primary"].value_counts()
    reference_counts.to_csv(
        OUT_DIR / "reference_primary_label_counts.csv"
    )

    print("\nReference primary-label counts:")
    print(reference_counts)

    # ========================================================
    # Sample sizes
    # ========================================================
    max_sample = min(500, n_total)

    sample_sizes = list(range(50, max_sample + 1, 50))

    all_results = []
    all_thresholds = []

    # ========================================================
    # Bootstrap + threshold perturbation
    # ========================================================
    for n_sample in sample_sizes:

        print(f"\nRunning sample size: {n_sample}/{n_total}")

        for b in range(N_BOOTSTRAP):

            sampled_idx = rng.choice(
                np.arange(n_total),
                size=n_sample,
                replace=False,
            )

            df_sample = df.iloc[sampled_idx].copy()

            for p in range(N_PERTURB):

                perturbed_labeled, thresholds = classify_with_threshold_reference(
                    df_all=df,
                    df_threshold_reference=df_sample,
                    rng=rng,
                    perturb_quantiles=True,
                    perturb_recovery=True,
                )

                perturbed_labels = perturbed_labeled["label_primary"].to_numpy()

                for primary_label in PRIMARY_ORDER:

                    mask = reference_labels == primary_label

                    if np.sum(mask) == 0:
                        retention = np.nan
                    else:
                        retention = np.mean(
                            perturbed_labels[mask] == reference_labels[mask]
                        )

                    all_results.append(
                        {
                            "n_sampled_paths": n_sample,
                            "sample_fraction": n_sample / n_total,
                            "bootstrap_id": b + 1,
                            "perturb_id": p + 1,
                            "primary_label": primary_label,
                            "retention_fraction": retention,
                        }
                    )

                threshold_row = {
                    "n_sampled_paths": n_sample,
                    "sample_fraction": n_sample / n_total,
                    "bootstrap_id": b + 1,
                    "perturb_id": p + 1,
                }
                threshold_row.update(thresholds)
                all_thresholds.append(threshold_row)

    all_results_df = pd.DataFrame(all_results)
    all_thresholds_df = pd.DataFrame(all_thresholds)

    all_results_df.to_csv(
        OUT_DIR / "bootstrap_threshold_all_retention_values.csv",
        index=False,
    )

    all_thresholds_df.to_csv(
        OUT_DIR / "bootstrap_threshold_all_thresholds.csv",
        index=False,
    )

    # ========================================================
    # Mean and std retention
    # ========================================================
    mean_df = (
        all_results_df
        .groupby(["n_sampled_paths", "sample_fraction", "primary_label"])["retention_fraction"]
        .mean()
        .reset_index()
    )

    std_df = (
        all_results_df
        .groupby(["n_sampled_paths", "sample_fraction", "primary_label"])["retention_fraction"]
        .std()
        .reset_index()
    )

    stability_mean_df = mean_df.pivot(
        index="n_sampled_paths",
        columns="primary_label",
        values="retention_fraction",
    )

    stability_std_df = std_df.pivot(
        index="n_sampled_paths",
        columns="primary_label",
        values="retention_fraction",
    )

    # Reorder columns
    stability_mean_df = stability_mean_df.reindex(columns=PRIMARY_ORDER)
    stability_std_df = stability_std_df.reindex(columns=PRIMARY_ORDER)

    stability_mean_df.to_csv(
        OUT_DIR / "combined_stability_mean_matrix.csv"
    )

    stability_std_df.to_csv(
        OUT_DIR / "combined_stability_std_matrix.csv"
    )

    return stability_mean_df, stability_std_df, all_results_df


# ============================================================
# OPTIONAL SUMMARY PRINTING
# ============================================================

def print_summary(stability_mean_df):
    """
    Prints a compact summary of convergence.
    """

    print("\nCombined bootstrap + threshold perturbation stability matrix:")
    print(stability_mean_df.round(3).to_string())

    first_row = stability_mean_df.iloc[0]
    last_row = stability_mean_df.iloc[-1]

    print("\nSmallest sample-size stability:")
    print(first_row.round(3).to_string())

    print("\nLargest sample-size stability:")
    print(last_row.round(3).to_string())

    print("\nMean stability over all classes:")
    mean_over_classes = stability_mean_df.mean(axis=1)
    print(mean_over_classes.round(3).to_string())


# ============================================================
# RUN ANALYSIS
# ============================================================

df = pd.read_csv(ORDER_DESCRIPTOR_FILE)

required_cols = [
    "pE",
    "E_dip",
    "C_dip",
    "rJ",
    "rC",
    "Lweak_E",
    "Lweak_C",
    "drop_E",
    "rise_E",
    "drop_C",
    "rise_C",
    "RE",
]

missing = [col for col in required_cols if col not in df.columns]

if missing:
    raise ValueError(f"Missing required descriptor columns: {missing}")

print("=" * 80)
print("Combined bootstrap + threshold perturbation reliability analysis")
print("=" * 80)
print(f"Input file: {ORDER_DESCRIPTOR_FILE}")
print(f"Number of pathways: {len(df)}")
print(f"N_BOOTSTRAP: {N_BOOTSTRAP}")
print(f"N_PERTURB: {N_PERTURB}")
print(f"Quantile perturbation width: ±{Q_PERTURB_WIDTH} percentile points")
print(f"Recovery perturbation width: ±{REC_PERTURB}")
print("=" * 80)

stability_mean_df, stability_std_df, all_results_df = run_combined_bootstrap_threshold_analysis(df)

print_summary(stability_mean_df)

### Save the stability matrices

In [ ]:
# Save the data frames used for plotting
stability_mean_df.to_csv(
    OUT_DIR / "combined_stability_mean_matrix.csv",
    index=True,
)
stability_std_df.to_csv(
    OUT_DIR / "combined_stability_std_matrix.csv",
    index=True,
)
print("Saved plot data frames to:")
print(OUT_DIR / "combined_stability_mean_matrix.csv")
print(OUT_DIR / "combined_stability_std_matrix.csv")

### Paper figure: combined stability heat map

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# GLOBAL PLOTTING SETTINGS
# ============================================================

plt.rcParams.update({
    "font.family": "sans-serif",  # good for papers
    "font.size": 11,               # base font size
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "axes.linewidth": 1.0,
    "xtick.major.width": 1.0,
    "ytick.major.width": 1.0,
    "xtick.major.size": 4,
    "ytick.major.size": 4,
})


# ============================================================
# COLUMN NAME → SHORT LABEL MAPPING
# ============================================================
# Maps the full column names in your dataframe to the short
# labels used on the x-axis of the heatmap.

COLUMN_LABEL_MAP = {
    "Source-limited": "SL",
    "GB-crossing (recovering)": "GBC",
    "GB-blocked (non-recovering)": "GBB",
    "Target-limited": "TL",
}


# ============================================================
# INPUT PATH
# ============================================================

OUT_DIR = Path("./results")  # TODO: point this at the folder where figures/CSVs from this notebook should be written

stability_mean_df = pd.read_csv(
    OUT_DIR / "combined_stability_mean_matrix.csv",
    index_col=0,
)

stability_std_df = pd.read_csv(
    OUT_DIR / "combined_stability_std_matrix.csv",
    index_col=0,
)


# ============================================================
# PLOTTING FUNCTION
# ============================================================

def plot_combined_stability_heatmap_square_figure_fill(
    stability_mean_df,
    stability_std_df=None,
    cmap="viridis",
    fig_size=8,
    out_name="combined_bootstrap_threshold_stability_heatmap_squarefill",
    show_std=True,
    xlabel="Primary labels",
    ylabel="Sampled pathways (N)",
    cbar_label="Primary-label retention fraction",
    column_label_map=COLUMN_LABEL_MAP,
):
    """
    Square overall figure + heatmap fills figure.
    Cells will NOT be square.

    show_std : bool
        If True (default), annotate each cell with mean ± std
        (when stability_std_df is provided). If False, only the
        mean value is shown, even if stability_std_df is provided.
    xlabel, ylabel, cbar_label : str or None
        Axis/colorbar titles. Pass None to omit any of them.
    column_label_map : dict or None
        Maps full column names in the dataframe (e.g. "Source-limited")
        to short x-axis tick labels (e.g. "SL"). Columns not found in
        the dict are shown unchanged. Pass None or {} to disable
        renaming and show the raw column names as-is.
    """

    plot_df = stability_mean_df.copy()
    nrows, ncols = plot_df.shape

    # ------------------------------------------------------------
    # Square figure
    # ------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(fig_size, fig_size))

    # ------------------------------------------------------------
    # Stronger contrast using actual data range
    # ------------------------------------------------------------
    data_vals = plot_df.values.astype(float)
    valid_vals = data_vals[~np.isnan(data_vals)]

    data_min = np.min(valid_vals)
    data_max = np.max(valid_vals)

    if data_max > data_min:
        pad = 0.01 * (data_max - data_min)
    else:
        pad = 0.01

    vmin = max(0.0, data_min - pad)
    vmax = min(1.0, data_max + pad)

    # ------------------------------------------------------------
    # Heatmap
    # aspect="auto" lets it fill the axes
    # ------------------------------------------------------------
    im = ax.imshow(
        plot_df.values,
        aspect="auto",
        vmin=vmin,
        vmax=vmax,
        cmap=cmap,
    )

    # ------------------------------------------------------------
    # Colorbar
    # ------------------------------------------------------------
    cbar = fig.colorbar(
        im,
        ax=ax,
        fraction=0.05,
        pad=0.04,
    )
    if cbar_label is not None:
        cbar.set_label(cbar_label, fontsize=plt.rcParams["axes.labelsize"])
    cbar.ax.tick_params(labelsize=plt.rcParams["ytick.labelsize"])

    # ------------------------------------------------------------
    # Titles and labels
    # ------------------------------------------------------------
    # ax.set_title(
    #     "Classification stability under bootstrap sampling and threshold perturbation",
    #     pad=14,
    # )

    if xlabel is not None:
        ax.set_xlabel(xlabel, labelpad=10)
    if ylabel is not None:
        ax.set_ylabel(ylabel, labelpad=10)

    # ------------------------------------------------------------
    # Ticks
    # ------------------------------------------------------------
    if column_label_map:
        xtick_labels = [column_label_map.get(c, c) for c in plot_df.columns]
    else:
        xtick_labels = list(plot_df.columns)

    ax.set_xticks(np.arange(ncols))
    ax.set_xticklabels(xtick_labels)
    ax.tick_params(axis='x', which='both', bottom=False, top=False)

    ax.set_yticks(np.arange(nrows))
    ax.set_yticklabels(plot_df.index)

    # ------------------------------------------------------------
    # Annotate cells
    # ------------------------------------------------------------
    midpoint = (vmin + vmax) / 2.0

    for i in range(nrows):
        for j in range(ncols):
            mean_val = plot_df.iloc[i, j]

            if np.isnan(mean_val):
                text = "NA"
                text_color = "black"
            else:
                if show_std and stability_std_df is not None:
                    std_val = stability_std_df.iloc[i, j]
                    if np.isnan(std_val):
                        text = f"{mean_val:.2f}"
                    else:
                        text = f"{mean_val:.2f}\n±{std_val:.2f}"
                else:
                    text = f"{mean_val:.2f}"

                text_color = "white" if mean_val < midpoint else "black"

            ax.text(
                j,
                i,
                text,
                ha="center",
                va="center",
                fontsize=plt.rcParams["font.size"] - 1,
                color=text_color,
            )

    # ------------------------------------------------------------
    # Layout adjustment
    # ------------------------------------------------------------
    fig.subplots_adjust(
        left=0.20,
        right=0.86,
        bottom=0.28,
        top=0.90,
    )

    # ------------------------------------------------------------
    # Save
    # ------------------------------------------------------------
    out_png = OUT_DIR / f"{out_name}.png"
    out_pdf = OUT_DIR / f"{out_name}.pdf"

    fig.savefig(out_png, bbox_inches="tight")
    fig.savefig(out_pdf, bbox_inches="tight")

    plt.show()

    print("\nSaved combined stability heat map:")
    print(out_png)
    print(out_pdf)


# ============================================================
# RUN
# ============================================================

plot_combined_stability_heatmap_square_figure_fill(
    stability_mean_df=stability_mean_df,
    stability_std_df=stability_std_df,
    cmap="viridis",   # or "PuOr"
    fig_size=8,
    show_std=False,   # set to False to hide ± std in cell annotations
    xlabel="Primary labels",
    ylabel="Sampled pathways (N)",
    cbar_label="Retention fraction",
)